# 🧪 SMILES & RDKit — Practice Exercises
### AI Literacy Project Taster Day — Queen Mary University of London

---

This notebook contains **hands-on exercises** to practise what you learned in the main SMILES tutorial.  
Each exercise has:
- 📖 A short explanation of the concept
- ✏️ A task for you to complete
- 💡 A hint if you get stuck
- ✅ A solution cell (hidden — try first before looking!)

> **How to use this notebook:**  
> Work through each exercise in order. Edit the cells marked ✏️ and press **Shift + Enter** to run them.  
> The solution cells are marked with `# ── SOLUTION ──` — try not to peek until you've had a go!

---

| Exercise | Topic | Difficulty |
|----------|-------|------------|
| 1 | Writing SMILES from scratch | ⭐ |
| 2 | Reading SMILES — identify the molecule | ⭐ |
| 3 | Drawing a set of common molecules | ⭐ |
| 4 | Rings and aromaticity | ⭐⭐ |
| 5 | Building a molecular property calculator | ⭐⭐ |
| 6 | Lipinski filter — find drug-like molecules | ⭐⭐ |
| 7 | Comparing molecules by similarity | ⭐⭐⭐ |
| 8 | Mystery molecule challenge | ⭐⭐⭐ |

---


## ⚙️ Setup — Run This First!

Run the cell below before starting any exercise. It imports all the tools you will need.


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdMolDescriptors
from rdkit import DataStructs
import pandas as pd
import matplotlib.pyplot as plt

print("✅ Ready! Now work through the exercises below.")


---
## Exercise 1 — Writing SMILES from Scratch ⭐

### 📖 Reminder: the key rules
- Atoms are written using their element symbol: `C`, `N`, `O`, `S`, `F`, `Cl`, `Br`
- Hydrogens bonded to organic atoms are **implicit** (you don't need to write them)
- Single bonds are implicit; double bonds = `=`, triple bonds = `#`
- Branches go in round brackets `(...)`
- Rings: break one bond and label both ends with the same number

**Trick of the trade:** https://pubchem.ncbi.nlm.nih.gov/

### ✏️ Your task

Write the SMILES for each molecule below and check your answer by drawing it with RDKit.  
The first one is done for you as an example.

**Molecules to encode:**

| # | Molecule | Formula | Hint |
|---|----------|---------|------|
| a | Methane | $CH_4$ | Just `C` — hydrogens are implicit |
| b | Ethane | $C_2H_6$ | Two carbons connected by a single bond |
| c | Ethene (ethylene) | $C_2H_4$ | Two carbons with a **double** bond |
| d | Ethyne (acetylene) | $C_2H_2$ | Two carbons with a **triple** bond |
| e | Methanol | $CH_3OH$ | A carbon connected to an oxygen |
| f | Propan-2-ol (isopropanol) | $(CH_3)_2CHOH$ | A carbon with two methyl **branches** and an OH |




In [ ]:
# ── Example: methane ──────────────────────────────────────────────────────────
methane = Chem.MolFromSmiles("C")
print("Methane drawn below:")
Draw.MolToImage(methane, size=(150, 150))


In [ ]:
# ✏️ Exercise 1 — fill in the SMILES strings below
# Replace each "???" with your answer

smiles_ethane     = "???"   # b: ethane
smiles_ethene     = "???"   # c: ethene
smiles_ethyne     = "???"   # d: ethyne
smiles_methanol   = "???"   # e: methanol
smiles_isopropanol= "???"   # f: propan-2-ol

# ── Draw them all in a grid ───────────────────────────────────────────────────
your_smiles = [smiles_ethane, smiles_ethene, smiles_ethyne,
               smiles_methanol, smiles_isopropanol]
labels      = ["b: Ethane", "c: Ethene", "d: Ethyne",
               "e: Methanol", "f: Propan-2-ol"]

mols = [Chem.MolFromSmiles(s) for s in your_smiles]

# Check for any invalid SMILES
for label, mol, smi in zip(labels, mols, your_smiles):
    if mol is None and smi != "???":
        print(f"❌ {label}: '{smi}' is not a valid SMILES — check your answer!")
    elif smi == "???":
        print(f"⏳ {label}: not filled in yet")
    else:
        print(f"✅ {label}: valid SMILES!")

valid_mols   = [m for m in mols if m is not None]
valid_labels = [l for m, l in zip(mols, labels) if m is not None]

if valid_mols:
    Draw.MolsToGridImage(valid_mols, molsPerRow=3,
                         subImgSize=(200, 200), legends=valid_labels)


<details>
<summary>💡 Hint (click to expand)</summary>

- Ethane: two carbons, single bond → `CC`
- Ethene: double bond between carbons → use `=`
- Ethyne: triple bond → use `#`
- Methanol: carbon + oxygen → `CO` (RDKit adds the H automatically)
- Propan-2-ol: central carbon has two CH₃ branches and one OH → `CC(C)O`

</details>

<details>
<summary>✅ Solution (try first!)</summary>

```python
smiles_ethane      = "CC"
smiles_ethene      = "C=C"
smiles_ethyne      = "C#C"
smiles_methanol    = "CO"
smiles_isopropanol = "CC(C)O"
```

</details>

---


## Exercise 2 — Reading SMILES: Identify the Molecule ⭐

### 📖 Concept
A key skill is being able to *read* a SMILES string and identify the molecule — or at least understand its key features — before drawing it.

### ✏️ Your task
For each SMILES below, **before running the code**:
1. Count the number of heavy atoms (non-hydrogen atoms)
2. Identify the functional groups present
3. Guess the molecule name if you can

Then run the cell to check your answer by drawing the molecule.


In [ ]:
# ── Mystery SMILES to identify ─────────────────────────────────────────────────
mystery_smiles = {
    "A": "CCO",                          # very simple — you should know this one!
    "B": "CC(=O)O",                      # hint: found in vinegar
    "C": "c1ccccc1",                     # hint: simplest aromatic compound
    "D": "CC(=O)Nc1ccc(O)cc1",           # hint: common painkiller
    "E": "OC(=O)c1ccccc1O",              # hint: used to make aspirin
    "F": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C" # hint: in your morning drink
}

# Draw all mystery molecules
mols   = [Chem.MolFromSmiles(s) for s in mystery_smiles.values()]
labels = list(mystery_smiles.keys())
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(220, 220), legends=labels)


✏️ **Fill in your guesses here** (just edit this cell — no code needed):

| SMILES label | My guess |
|---|---|
| A | ? |
| B | ? |
| C | ? |
| D | ? |
| E | ? |
| F | ? |

<details>
<summary>✅ Answers (try first!)</summary>

| Label | Molecule | Key features |
|-------|----------|-------------|
| A | Ethanol | C–C–O chain |
| B | Acetic acid | C–C(=O)–O (carboxylic acid) |
| C | Benzene | Aromatic ring (lowercase `c`) |
| D | Paracetamol (acetaminophen) | Amide group + para-substituted benzene ring + OH |
| E | Salicylic acid | Benzene ring + carboxylic acid + OH (aspirin precursor) |
| F | Caffeine | Two fused rings with multiple N atoms |

</details>

---


## Exercise 3 — Drawing Medicines You've Heard Of ⭐

### 📖 Concept
Many common medicines have known SMILES strings — pharmaceutical databases like PubChem store them all.  
Let's draw a "medicine cabinet" of well-known drugs.

### ✏️ Your task
The SMILES for five common medicines are given below. Run the cell to draw them, then answer the questions underneath.


In [ ]:
# ── A medicine cabinet ─────────────────────────────────────────────────────────
medicines = {
    "Aspirin"     : "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen"   : "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Paracetamol" : "CC(=O)Nc1ccc(O)cc1",
    "Penicillin G": "CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O",
    "Caffeine"    : "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
}

mols   = [Chem.MolFromSmiles(s) for s in medicines.values()]
labels = list(medicines.keys())
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(280, 280), legends=labels)


In [ ]:
# ✏️ Count the number of rings in each molecule
# Hint: use rdMolDescriptors.CalcNumRings(mol)

for name, smiles in medicines.items():
    mol   = Chem.MolFromSmiles(smiles)
    rings = rdMolDescriptors.CalcNumRings(mol)
    # ── fill in the blank below ──────────────────────────────────
    print(f"{name:<15}: {rings} ring(s)")


> 💬 **Questions:**
> 1. Which medicine has the most rings? Why do you think complex ring systems are common in drugs?
> 2. Penicillin G contains a four-membered ring (the β-lactam). Can you spot it in the structure?
> 3. Aspirin and salicylic acid (Exercise 2E) are closely related — what is the chemical difference?

---


## Exercise 4 — Rings and Aromaticity ⭐⭐

### 📖 Concept: Aromatic vs Non-Aromatic Rings

In SMILES, aromatic atoms are written in **lowercase** (`c`, `n`, `o`), while non-aromatic atoms are in **uppercase** (`C`, `N`, `O`).

- `C1CCCCC1` = **cyclohexane** (non-aromatic, all single bonds)
- `c1ccccc1`  = **benzene** (aromatic, delocalised electrons)

RDKit draws these very differently!

### ✏️ Task A — spot the difference


In [ ]:
# ── Draw cyclohexane vs benzene side by side ──────────────────────────────────
ring_molecules = {
    "Cyclohexane\n(non-aromatic)": "C1CCCCC1",
    "Benzene\n(aromatic)"        : "c1ccccc1",
    "Cyclohexene\n(one double bond)": "C1=CCCCC1",
    "Naphthalene\n(two fused rings)" : "c1ccc2ccccc2c1",
}

mols   = [Chem.MolFromSmiles(s) for s in ring_molecules.values()]
labels = list(ring_molecules.keys())
Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(220, 220), legends=labels)


> 💬 Notice how RDKit draws the aromatic benzene ring with a circle, while cyclohexane uses a hexagon with explicit single bonds.

### ✏️ Task B — write the SMILES for these ring compounds


In [ ]:
# ✏️ Write SMILES for the following ring compounds
# Replace "???" with your answer

smiles_cyclopentane  = "???"  # 5-membered non-aromatic ring (all CH₂)
smiles_pyridine      = "???"  # benzene ring where ONE carbon is replaced by nitrogen (N)
smiles_phenol        = "???"  # benzene ring with an OH group attached

your_rings = {
    "Cyclopentane" : smiles_cyclopentane,
    "Pyridine"     : smiles_pyridine,
    "Phenol"       : smiles_phenol,
}

mols   = [Chem.MolFromSmiles(s) for s in your_rings.values()]
labels = list(your_rings.keys())

for name, mol, smi in zip(labels, mols, your_rings.values()):
    if smi == "???":
        print(f"⏳ {name}: not filled in yet")
    elif mol is None:
        print(f"❌ {name}: '{smi}' is not valid SMILES")
    else:
        print(f"✅ {name}: valid!")

valid_mols   = [m for m in mols if m is not None]
valid_labels = [l for m, l in zip(mols, labels) if m is not None]
if valid_mols:
    Draw.MolsToGridImage(valid_mols, molsPerRow=3,
                         subImgSize=(220, 220), legends=valid_labels)


<details>
<summary>💡 Hint</summary>

- Cyclopentane: like cyclohexane but with 5 carbons → `C1CCCC1`
- Pyridine: replace one `c` in benzene with `n` (aromatic nitrogen)
- Phenol: benzene ring (`c1ccccc1`) with `O` attached to one carbon

</details>

<details>
<summary>✅ Solution</summary>

```python
smiles_cyclopentane = "C1CCCC1"
smiles_pyridine     = "c1ccncc1"
smiles_phenol       = "Oc1ccccc1"
```

</details>

---


## Exercise 5 — Build Your Own Molecular Property Calculator ⭐⭐

### 📖 Concept
In the main tutorial, we calculated molecular properties one at a time.  
Now you'll build a **reusable function** that calculates all properties at once for any SMILES you give it.

In Python, a **function** is a block of code you can call by name and reuse. Here's the structure:

```python
def my_function(input_value):
    # do something with input_value
    result = input_value * 2
    return result
```

### ✏️ Your task
Complete the function below by filling in the missing property calculations.


In [ ]:
def molecular_report(smiles: str) -> None:
    """
    Print a summary of molecular properties for a given SMILES string.

    Parameters
    ----------
    smiles : str — a valid SMILES string
    """
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print(f"❌ '{smiles}' is not a valid SMILES string.")
        return

    # ── Calculate properties ───────────────────────────────────────────────────
    mw   = rdMolDescriptors.CalcExactMolWt(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)       # H-bond donors
    hba  = rdMolDescriptors.CalcNumHBA(mol)       # H-bond acceptors
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    rings= rdMolDescriptors.CalcNumRings(mol)
    arom = rdMolDescriptors.CalcNumAromaticRings(mol)

    # ── Lipinski pass/fail ─────────────────────────────────────────────────────
    lipinski_pass = (mw <= 500) and (hbd <= 5) and (hba <= 10) and (tpsa < 140)

    # ── Print report ──────────────────────────────────────────────────────────
    print(f"{'='*45}")
    print(f"  Molecular Report for: {smiles}")
    print(f"{'='*45}")
    print(f"  Molecular weight  : {mw:.2f} Da   {'✅' if mw <= 500  else '❌'} (limit ≤ 500)")
    print(f"  H-bond donors     : {hbd}          {'✅' if hbd <= 5   else '❌'} (limit ≤ 5)")
    print(f"  H-bond acceptors  : {hba}          {'✅' if hba <= 10  else '❌'} (limit ≤ 10)")
    print(f"  TPSA              : {tpsa:.1f} Å²  {'✅' if tpsa < 140 else '❌'} (limit < 140)")
    print(f"  Total rings       : {rings}")
    print(f"  Aromatic rings    : {arom}")
    print(f"{'─'*45}")
    print(f"  Lipinski Rule of Five: {'✅ PASS — likely drug-like' if lipinski_pass else '❌ FAIL'}")
    print(f"{'='*45}")

    return Draw.MolToImage(mol, size=(300, 300))

# ── Test it on ibuprofen ───────────────────────────────────────────────────────
molecular_report("CC(C)Cc1ccc(cc1)C(C)C(=O)O")


In [ ]:
# ✏️ Now test the function on THREE molecules of your choice
# Try a drug you know, or make up a molecule using SMILES rules!

# Replace the SMILES strings below:
molecular_report("???")   # your first molecule


In [ ]:
molecular_report("???")   # your second molecule


In [ ]:
molecular_report("???")   # your third molecule


> 💬 **Challenge:** Can you find a molecule that **fails** the Lipinski filter?  
> Hint: try a very large molecule or something with many OH groups.

---


## Exercise 6 — Which of These Are Drug-Like? ⭐⭐

### 📖 Concept
Pharmaceutical companies use Lipinski's Rule of Five to filter millions of compounds down to a manageable shortlist for lab testing.

Below is a small dataset of 10 compounds — some drug-like, some not. Your task is to apply the Lipinski filter and find out which ones pass.

### ✏️ Your task
Run the cell to compute all descriptors, then answer the questions below.


In [ ]:
# ── A mixed dataset of 10 compounds ───────────────────────────────────────────
compound_data = {
    "Paracetamol"       : "CC(=O)Nc1ccc(O)cc1",
    "Ibuprofen"         : "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Caffeine"          : "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Aspirin"           : "CC(=O)Oc1ccccc1C(=O)O",
    "Glucose"           : "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O",
    "Taxol (Paclitaxel)": "CC1=C2[C@@]([C@H]([C@@H]3[C@]4([C@@H](OC4=O)[C@@H](O)[C@@H]([C@@H]34)OC(=O)c3ccccc3)OC(C)=O)(CC[C@@H]2OC(=O)[C@@H](NC(=O)c2ccccc2)[C@@H](O)c2ccccc2)O)(C(C)=O)OC(=O)C1",
    "Vancomycin"        : "CC[C@H](C)[C@@H]1NC(=O)[C@@H](Cc2cc(Cl)c(O[C@@H]3C[C@](N)(C(=O)O)[C@@H](O)[C@@H](O3)c3cc(Cl)c(O[C@H]4O[C@@H](CO)[C@@H](O)[C@H](O)[C@@H]4NC(C)=O)c(O)c3)c(O)c2)[NH2+][C@H](C(=O)N[C@@H](CC(N)=O)C(=O)N[C@@H]2[C@@H](O)c3ccc(Oc4cc5cc(Oc6cc(C(=O)N[C@H](C(=O)N1)c1ccc(O)cc1Cl)c(O)c(c1)[C@@H](NC2=O)C(=O)O)c(O)c5c(Cl)c4)c(Cl)c3)c1ccc(O)cc1",
    "Cholesterol"       : "CC(C)CCC[C@@H](C)[C@H]1CC[C@H]2[C@@H]3CC=C4C[C@@H](O)CC[C@]4(C)[C@H]3CC[C@]12C",
    "Ethanol"           : "CCO",
    "Penicillin G"      : "CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O",
}

# ── Compute Lipinski descriptors for each ─────────────────────────────────────
rows = []
for name, smiles in compound_data.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        mw   = rdMolDescriptors.CalcExactMolWt(mol)
        hbd  = rdMolDescriptors.CalcNumHBD(mol)
        hba  = rdMolDescriptors.CalcNumHBA(mol)
        tpsa = rdMolDescriptors.CalcTPSA(mol)
        passes = (mw <= 500) and (hbd <= 5) and (hba <= 10) and (tpsa < 140)
        rows.append({
            "Compound"  : name,
            "MW (Da)"   : round(mw, 1),
            "HBD"       : hbd,
            "HBA"       : hba,
            "TPSA (Å²)" : round(tpsa, 1),
            "Lipinski ✅/❌": "✅ Pass" if passes else "❌ Fail",
        })

results_df = pd.DataFrame(rows).set_index("Compound")
results_df


> 💬 **Questions:**
> 1. How many compounds passed the Lipinski filter?
> 2. Taxol (paclitaxel) is a widely used cancer drug but **fails** the filter. Look up its molecular weight — why does it violate Rule of Five? Does this mean it's a bad drug?
> 3. Vancomycin is a critical antibiotic. Does it pass? What does this tell you about the Rule of Five?
> 4. Ethanol passes — does that mean it's a good drug candidate? What does this reveal about the *limitations* of the Lipinski filter?

---


## Exercise 7 — Molecular Similarity: Find the Closest Drug ⭐⭐⭐

### 📖 Concept
In the main tutorial, you found that **nilotinib** is the most similar FDA-approved drug to imatinib.  
Now you'll repeat this analysis with a **different reference molecule** of your choice.

### ✏️ Your task
1. Choose any molecule as your reference (a drug, a molecule from a previous exercise, or anything you like)
2. Build a small comparison library from the medicines in Exercise 3
3. Rank them by Tanimoto similarity to your reference molecule


In [ ]:
# ── Your reference molecule ────────────────────────────────────────────────────
# ✏️ Replace the SMILES below with any molecule you want to use as a reference
reference_smiles = "CC(=O)Oc1ccccc1C(=O)O"   # default: aspirin
reference_name   = "Aspirin"                   # give it a name

reference_mol = Chem.MolFromSmiles(reference_smiles)
print(f"Reference molecule: {reference_name}")
Draw.MolToImage(reference_mol, size=(250, 250))


In [ ]:
# ── Comparison library ─────────────────────────────────────────────────────────
comparison_library = {
    "Ibuprofen"        : "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Paracetamol"      : "CC(=O)Nc1ccc(O)cc1",
    "Caffeine"         : "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Salicylic acid"   : "OC(=O)c1ccccc1O",
    "Penicillin G"     : "CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O",
    "Benzene"          : "c1ccccc1",
    "Ethanol"          : "CCO",
    "Glucose"          : "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O",
    "Imatinib"         : "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5",
    "Cholesterol"      : "CC(C)CCC[C@@H](C)[C@H]1CC[C@H]2[C@@H]3CC=C4C[C@@H](O)CC[C@]4(C)[C@H]3CC[C@]12C",
}

# ── Calculate Tanimoto similarity to reference ────────────────────────────────
ref_fp = Chem.RDKFingerprint(reference_mol)

similarity_rows = []
for name, smiles in comparison_library.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        fp  = Chem.RDKFingerprint(mol)
        sim = DataStructs.FingerprintSimilarity(ref_fp, fp)
        similarity_rows.append({"Compound": name, "Tanimoto Similarity": round(sim, 4)})

sim_df = (pd.DataFrame(similarity_rows)
            .set_index("Compound")
            .sort_values("Tanimoto Similarity", ascending=False))

print(f"Similarity ranking vs. {reference_name}:\n")
print(sim_df.to_string())


In [ ]:
# ── Visualise the top 4 most similar compounds ─────────────────────────────────
top4_names  = sim_df.head(4).index.tolist()
top4_smiles = [comparison_library[n] for n in top4_names]
top4_sims   = sim_df.head(4)["Tanimoto Similarity"].tolist()

top4_mols   = [Chem.MolFromSmiles(s) for s in top4_smiles]
top4_labels = [f"{n}\n(T={s:.3f})" for n, s in zip(top4_names, top4_sims)]

Draw.MolsToGridImage(top4_mols, molsPerRow=4,
                     subImgSize=(250, 250), legends=top4_labels)


> 💬 **Questions:**
> 1. Which compound was most similar to your reference? Does that make chemical sense?
> 2. Change the reference molecule to **ibuprofen** — does the ranking change significantly?
> 3. What is the Tanimoto similarity between aspirin and salicylic acid? Why are they so similar?
> 4. What is the similarity between aspirin and ethanol? What does a value near 0 mean structurally?

---


## Exercise 8 — Mystery Molecule Challenge ⭐⭐⭐

### 📖 The challenge
You are given **four mystery SMILES**. For each one:
1. Draw the molecule
2. Calculate its molecular properties using your `molecular_report()` function from Exercise 5
3. Use the Tanimoto similarity to compare it against the medicines from Exercise 3
4. Based on all of the above, try to identify the molecule

Clues are available if you need them!


In [ ]:
# ── Four mystery molecules ─────────────────────────────────────────────────────
mysteries = {
    "Mystery 1": "c1ccc2[nH]cccc2c1",
    "Mystery 2": "CC(=O)OCC1=C(N2C(=O)[C@@H]([NH3+])[C@H]2SC1)C(=O)O",
    "Mystery 3": "CN(C)c1ccc(cc1)N=Nc1ccccc1S(=O)(=O)[O-]",
    "Mystery 4": "OC(=O)[C@@H](N)Cc1ccc(O)cc1",
}

mols   = [Chem.MolFromSmiles(s) for s in mysteries.values()]
labels = list(mysteries.keys())
Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(260, 260), legends=labels)


In [ ]:
# ✏️ Use your molecular_report() function on each mystery molecule
# Then discuss with your partner what each might be

for name, smiles in mysteries.items():
    print(f"\n{'▶'*3} {name}")
    molecular_report(smiles)


In [ ]:
# ── Compare mysteries to known medicines using Tanimoto similarity ─────────────
known_medicines = {
    "Aspirin"     : "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen"   : "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Paracetamol" : "CC(=O)Nc1ccc(O)cc1",
    "Penicillin G": "CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O",
    "Caffeine"    : "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
}

print("Tanimoto Similarity Matrix (Mystery vs Known Medicines)\n")
header = f"{'':>12}" + "".join(f"{k:>14}" for k in known_medicines)
print(header)
print("─" * len(header))

for mname, msmiles in mysteries.items():
    mol    = Chem.MolFromSmiles(msmiles)
    mfp    = Chem.RDKFingerprint(mol)
    row    = f"{mname:>12}"
    for ksmiles in known_medicines.values():
        kmol = Chem.MolFromSmiles(ksmiles)
        kfp  = Chem.RDKFingerprint(kmol)
        sim  = DataStructs.FingerprintSimilarity(mfp, kfp)
        row += f"{sim:>14.3f}"
    print(row)


<details>
<summary>💡 Clues (click to expand)</summary>

- **Mystery 1:** Found in plants; a heterocyclic compound with a fused ring system containing nitrogen. Common in biology.
- **Mystery 2:** A β-lactam antibiotic — notice the 4-membered ring. Related to but different from penicillin.
- **Mystery 3:** A synthetic dye used in staining biological samples for microscopy. Contains an azo group (N=N).
- **Mystery 4:** An amino acid found in proteins. Its name comes from the Greek word for "cheese" (τυρός).

</details>

<details>
<summary>✅ Answers</summary>

| Mystery | Identity | Key features |
|---------|----------|-------------|
| 1 | Indole | Fused benzene + pyrrole ring; present in tryptophan |
| 2 | Cephalexin | First-generation cephalosporin antibiotic |
| 3 | Methyl orange | Azo dye (N=N); pH indicator |
| 4 | Tyrosine | Amino acid with a para-hydroxyphenyl group |

</details>

---


## 🌟 Bonus — Design Your Own Molecule

Now that you understand SMILES, try to **design a molecule** that:
1. Passes Lipinski's Rule of Five (drug-like)
2. Contains at least one aromatic ring
3. Contains at least one functional group from the list below

| Functional group | SMILES fragment |
|-----------------|----------------|
| Carboxylic acid | `C(=O)O` |
| Amine           | `N` |
| Amide           | `C(=O)N` |
| Hydroxyl (alcohol) | `O` |
| Ester           | `C(=O)OC` |
| Halogen         | `F`, `Cl`, `Br` |


In [ ]:
# ✏️ Design your own molecule here!
# Start simple — a substituted benzene ring is a good starting point

my_molecule_smiles = "???"   # replace with your SMILES
my_molecule_name   = "My molecule"

mol = Chem.MolFromSmiles(my_molecule_smiles)

if mol is None:
    print("❌ Invalid SMILES — check your string and try again.")
elif my_molecule_smiles == "???":
    print("⏳ Fill in your SMILES above!")
else:
    print(f"🎉 You designed: {my_molecule_name}")
    molecular_report(my_molecule_smiles)


> 💬 Share your molecule with the group! Can your neighbour read your SMILES and guess what you designed before seeing the drawing?

---

## 🎓 Well Done!

You have completed all the SMILES practice exercises. Here's what you can now do:

- ✅ Write SMILES for simple and complex molecules
- ✅ Read a SMILES string and identify key features
- ✅ Use RDKit to draw molecules and calculate their properties
- ✅ Apply Lipinski's Rule of Five to filter drug-like compounds
- ✅ Measure molecular similarity using Tanimoto fingerprints
- ✅ Build a reusable Python function for molecular analysis

---
*Exercises developed for the QMUL AI Literacy Project Taster Day · Department of Chemistry*
